# Phase 6.4 — momentum feature comparison

This notebook performs the first actual Phase 6 feature experiment. It compares the full approved market-feature set with exactly the same set **without the three momentum columns**. It does this separately for return and risk, across the same four historical time folds.

It fits models. Locally, select the project `.venv` kernel. In Colab, the setup cell clones the repository, downloads hash-verified release inputs, and installs the pinned dependencies. It creates a temporary evidence folder and refuses to overwrite an old one. Download the evidence before ending the Colab runtime. It does not claim a final trading result or an untouched test result.

In [ ]:
from pathlib import Path
import subprocess
import sys

BRANCH = 'feature/model-evaluation'
COLAB_REPOSITORY = Path('/content/HalalifyAI')
if 'google.colab' in sys.modules:
    if not (COLAB_REPOSITORY / '.git').is_dir():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/frager098/HalalifyAI.git', str(COLAB_REPOSITORY)], check=True)
    root = COLAB_REPOSITORY.resolve()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt'), '-r', str(root / 'requirements-cleaning.txt'), 'gdown>=5.2,<6'], check=True)
else:
    root = Path.cwd().resolve()
    while root != root.parent and not (root / 'src' / 'model_training').is_dir():
        root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.notebook_environment import ensure_phase5_dataset, prepare_repository
root = prepare_repository(branch=BRANCH, colab_repository=COLAB_REPOSITORY)
ensure_phase5_dataset(root)
%cd {root}
print(f'Project folder: {root}')

## Run the controlled comparison

The next cell uses only the exact dataset and chronological protocol continued from Phase 5. It uses HistGradientBoosting, tunes only on the earlier inner-validation period, and scores each later outer fold once. Expect a few minutes of work. Do not stop the kernel while it is running.

In [ ]:
from src.model_training.feature_ablation.phase5_continuation import (
    MOMENTUM_OUTPUT,
    prepare_inputs,
    write_momentum_report,
)

prepare_inputs()
report = write_momentum_report()
print('Status:', report['status'])
print('Evidence folder:', MOMENTUM_OUTPUT)
for task, summary in report['task_summaries'].items():
    print(f"{task}: {summary['decision']} | {summary['primary_metric']} difference = {summary['mean_difference']:.4f}")

## Read the result

`retain` means momentum improved the pre-defined main score on average and in at least three of the four folds. `reject` means it did not help in most folds. `inconclusive` means the evidence was mixed. For risk, a result cannot be retained if it makes High-risk detection worse or creates more serious High-to-Low mistakes.

In [ ]:
for task, summary in report['task_summaries'].items():
    print('\n', task.upper())
    display(summary['fold_metrics'])
    if task == 'risk':
        print('Risk safety checks:', summary['risk_guardrails'])